<a href="https://colab.research.google.com/github/Bilkouristas/Benchmark/blob/master/RelativePoseEstimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np


In [3]:
def create_rotation_matrix(rx,ry,rz):
  Rx = np.array([1,0,0],
                [0, np.cos(rx) , -np.sin(rx)],
                [0, np.sin(rx), np.cos(rx)])
  Ry = np.array([np.cos(ry),0,np.sin(ry)],
                [0, 1, 0],
                [-np.sin(ry), 0, np.cos(ry)] )
  Rz = np.array([np.cos(rz), -np.sin(rz), 0],
                [np.sin(rz), np.cos(rz), 0],
                [0, 0, 1])
  return Rz @ Ry @ Rx

def build_linear_system(X,x):
  N = X.shape[0]
  A = []

  for i in range(N):
    X_i, Y_i, Z_i = X[i,:]
    u_i, v_i = x[i,:]

    A.append([X_i, Y_i, Z_i, 0, 0, 0, -u_i*X_i, -u_i*Y_i, -u_i*Z_i])
    A.append([0, 0 ,0, X_i, Y_i, Z_i, -v_i*X_i, -v_i*Y_i, -v_i*Z_i])

  return np.array(A)


def solve_linear_system(X, x):
  A = build_linear_system(X,x)


  _,_,V = np.linalg.svd(A)
  p = V[-1]
  P = p.reshape(3,4)

  R = P[:, :3]
  t = P[:, 3]

  column_norm = np.linalg.norm(R, axis=0)
  R = R / column_norm
  t = t / column_norm
  return R,t

  # U, S, V = np.linalg.svd(A)
  #

def project_points(R,t,X):
    X_cam = (R @ X.T) + t
    X_cam = X_cam.T
    x_2d = X_cam[:, 0] / X_cam[:, 2]
    y_2d = X_cam[:, 1] / X_cam[:, 2]
    X_cam = np.stack((x_2d, y_2d), axis=1)

    return X_cam


if __name__ == "__main__":
  np.random.seed(0)

  N = 8
  X_3D = np.random.rand(N,3)

  R_true = create_rotation_matrix(np.radians(30), np.radians(15), np.radians(60))
  t_true = np.array([1.0,2.0,3.0])

  x_2D = project_points(R_true, t_true, X_3D)
  x_2D_noisy = x_2D + np.random.randn(N,2) * 0.01

  R_est = solve_linear_system(X_3D, x_2D_noisy)
  print(R_est)
  print(R_true)






TypeError: array() takes from 1 to 2 positional arguments but 3 were given